In [1]:
import math
import sys
from pathlib import Path

toolbox_path = Path.cwd() / "support_material" / "input" / "exams"
if str(toolbox_path) not in sys.path:
    sys.path.insert(0, str(toolbox_path))

from exam_tools import *
import exam_tools
from sympy import *

print(f"Klar: {len(exam_tools.__all__)} offentlige funktioner")

Klar: 12 offentlige funktioner


#### Hurtigt workflow

1. Skriv givne værdier og konverter enheder (`L -> m3`, `°C -> K`, `mg -> g`).
2. Afstem/fortolk reaktionen manuelt, når opgaven kræver det.
3. Vælg en funktion eller en kort sammensætning nedenfor.
4. Sammenlign resultat, fortegn og størrelsesorden med svarmulighederne.

#### Stofmængde, gas og materialer

| Funktion | Brug når | Input -> output / enheder | Minimal kald | Eksamenseksempel |
|---|---|---|---|---|
| `molar_mass` | en formel skal omsættes til molarmasse | formel -> `g/mol` | `molar_mass("NaBr")` | `molar_mass("Ca3N2")` |
| `ideal_gas` | én af `p,V,n,T` mangler | Pa, `m3`, mol, K -> manglende SI-enhed | `ideal_gas(pressure_pa=101325, volume_m3=1e-3, temperature_k=298.15)` | `ideal_gas(pressure_pa=101325, amount_mol=5/molar_mass("Ca3N2"), temperature_k=298.15)` |
| `unit_cell_volume` | celledensitet/krystalvolumen | `kg/m3`, `g/mol`, partikler pr. celle -> `m3` | `unit_cell_volume(7874, 55.85, 2)` | BCC-jern: `unit_cell_volume(7874, 55.85, 2)` |

#### Termodynamik, ligevægt og elektrokemi

| Funktion | Brug når | Input -> output / enheder | Minimal kald | Eksamenseksempel |
|---|---|---|---|---|
| `equilibrium_constant` | standard `ΔG` skal omsættes til `K` | `kJ/mol`, K -> dimensionsløs `K` | `equilibrium_constant(-10, 298.15)` | ammoniaksyntese efter `ΔG` |
| `cell_potential` | Nernstpotentiale ønskes | `E°` i V, elektroner, `Q`, K -> V | `cell_potential(0, 2, 0.01)` | Ni-koncentrationscelle |

#### Opløsninger og faseændringer

| Funktion | Brug når | Input -> output / enheder | Minimal kald | Eksamenseksempel |
|---|---|---|---|---|
| `weak_solution_ph` | svag monoprot syre/base | koncentration `M`, `Ka`/`Kb`, type -> pH | `weak_solution_ph(0.1, 1.8e-5, "base")` | ammoniak eller benzoesyre |
| `solubility_from_ksp` | molær opløselighed eller fælles-ion | `Ksp`, ionkoefficienter, evt. baggrunds-`M` -> `M` | `solubility_from_ksp(1e-10, (1,1))` | `Mg(OH)2`: `solubility_from_ksp(5.61e-12,(1,2))` |
| `solubility_complete_check` | opløses en tilsat saltmasse helt? | formel, `Ksp`, masse, volumen, fælles-ion -> resultatdict | `solubility_complete_check("AgBr",3.3e-13,added_mass_mg=10,common_ion_concentration_m=1e-6)` | AgBr mod CuBr |
| `freezing_point_depression` | frysepunktssænkning | mol eller `g`/`g/mol`, solvent `kg`, `i` -> resultatdict | `freezing_point_depression(solute_mol=.1, solvent_mass_kg=.5)` | NaCl mod CaCl2 |
| `clausius_clapeyron_pressure` | damptryk ved ny temperatur | Pa, K, K, `kJ/mol` -> Pa | `clausius_clapeyron_pressure(101325,300,350,20)` | ethan-kogepunkt ved højere tryk |

#### Kinetik

| Funktion | Brug når | Input -> output / enheder | Minimal kald | Eksamenseksempel |
|---|---|---|---|---|
| `kinetic_linear_fit` | tidsserie skal testes mod 0./1./2. orden | tider `s`, koncentrationer `M`, orden -> `(hældning, skæring, R²)` | `kinetic_linear_fit([0,1],[1,.5],1)` | iodid-koncentration mod tid |
| `arrhenius_ratio` | temperaturændring og `Ea` giver hastighedsforhold | `kJ/mol`, K, K -> `k2/k1` | `arrhenius_ratio(50,300,350)` | 600 °C mod 800 °C eller kandidat-`Ea` |

In [2]:
H1 = -393.5
H2 =-285.8
H3 =-1300

-H3+H2+2*H1

227.20000000000005

In [3]:
molar_mass("NH3"), molar_mass("HBr")

(17.031, 80.91199999999999)

In [4]:
tripleC = -835
singleC = -350
CH = -410
HH = -436

- (tripleC + 2*HH) + (singleC + 4*CH)

-283

In [5]:
NaCl = freezing_point_depression(
    solute_mass_g=1.0, molar_mass_g_mol=58.44,
    solvent_mass_kg=0.500, vant_hoff_factor=1.8,
)
CaCl2 = freezing_point_depression(
    solute_mass_g=1.0, molar_mass_g_mol=110.98,
    solvent_mass_kg=0.500, vant_hoff_factor=2.6,
)

NaCl, CaCl2

({'solute_mol': 0.017111567419575632,
  'molality_mol_kg': 0.034223134839151265,
  'vant_hoff_factor': 1.8,
  'delta_tf_c': 0.11457905544147845,
  'freezing_point_c': -0.11457905544147845},
 {'solute_mol': 0.009010632546404758,
  'molality_mol_kg': 0.018021265092809515,
  'vant_hoff_factor': 2.6,
  'delta_tf_c': 0.08715083798882682,
  'freezing_point_c': -0.08715083798882682})

In [11]:
AgBr = solubility_complete_check(
    "AgBr", ksp=3.3e-13, added_mass_mg=10,
    solution_volume_l=1.0, common_ion_concentration_m=1.0e-6,
    stoich_metal=1, stoich_common_ion=1,
)
CuBr = solubility_complete_check(
    "CuBr", ksp=5.3e-9, added_mass_mg=100,
    solution_volume_l=1.0, common_ion_concentration_m=1.0e-6,
    stoich_metal=1, stoich_common_ion=1,
)

print("AgBr:", AgBr)
print("CuBr:", CuBr)
print("AgBr opløses ikke fuldstændigt." if not AgBr["dissolves_completely"] else "AgBr opløses fuldstændigt.")
print("CuBr opløses fuldstændigt." if CuBr["dissolves_completely"] else "CuBr opløses ikke fuldstændigt.")